# Lab 7
- **Student Name**: Daniel Alejandro Castillo Martin
- **Student ID**: N10019284
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
---

## Lab Overview

This lab implements and verifies a variational autoencoder (VAE) on 8 × 8 digit images. It compares the VAE with a plain autoencoder, measures how the KL weight affects reconstruction and prior sampling, and evaluates latent interpolation, class conditioning, and generated samples.

## 1. Imports and Setup

In [12]:
# Import required libraries
import torch
from torch import nn
from torch.nn import functional as F

# Import the starter code
from scripts import lab07_starter as S

# Import custom modules
from src.vae import VAE, vae_loss, kl_to_standard_normal, reconstruction_nll

# Load the data
x_train, y_train, x_test, y_test = S.load_task(seed = 0)

In [6]:
# Define the dimensions of the model as constants
INPUT_DIM = 64
HIDDEN_DIM = 64
LATENT_DIM = 8

# Print the shapes of the training and test data
print(f'Training: {tuple(x_train.shape)} | Test: {tuple(x_test.shape)}')

Training: (1438, 64) | Test: (359, 64)


## 2. VAE Implementation and Verification

### 2.2 VAE Architecture

The encoder maps each 64-pixel image to the mean and log-variance of an 8-dimensional Gaussian. Reparameterization combines those values with standard normal noise so reconstruction gradients can reach the encoder. The decoder maps the sampled latent vector back to 64 pixel logits.

In [7]:
# Set the seed for reproducibility
torch.manual_seed(0)

# Initialize the VAE
vae = VAE(input_dim = INPUT_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM)

# Forward pass through the VAE
with torch.no_grad():
    logits, mu, logvar = vae(x_train[:4])

# Print the shapes of the decoder logits and the posterior mean and log-variance
print(f'Decoder logits: {tuple(logits.shape)}')
print(f'Posterior mean and log-variance: {tuple(mu.shape)}')

Decoder logits: (4, 64)
Posterior mean and log-variance: (4, 8)


### 2.3 VAE Loss

The VAE loss combines reconstruction error with a KL penalty. The reconstruction term sums binary cross-entropy over the 64 pixels. The KL term sums the divergence between the encoder’s Gaussian and the standard normal over the 8 latent dimensions. The weighted sum is averaged over the batch. The parameter $\beta$ controls the weight of the KL term.

In [8]:
# Calculate the initial VAE loss
with torch.no_grad():
    loss = vae_loss(vae, x_train[:4], beta = 1.0)

print(f'Initial VAE loss: {loss.item():.4f}')

Initial VAE loss: 44.3535


### 2.4 Closed-Form KL vs. Monte Carlo Estimate

For five randomly generated diagonal Gaussians, this check compares the closed-form KL divergence to a standard normal with a Monte Carlo estimate. The estimate averages $log q(z) - \log p(z)$ over samples drawn from $q$. Small differences are expected because the estimate uses a finite number of samples.

In [ ]:
generator = torch.Generator().manual_seed(0)
n_gaussians = 5
n_samples = 100_000
shape = (n_gaussians, LATENT_DIM)

# Define five diagonal Gaussians and the standard normal prior
mu_check = torch.randn(shape, generator = generator)
logvar_check = 0.5 * torch.randn(shape, generator = generator)
posterior = torch.distributions.Normal(mu_check, torch.exp(0.5 * logvar_check))
prior = torch.distributions.Normal(torch.zeros_like(mu_check), torch.ones_like(mu_check))

# Draw samples from each posterior and sum log densities over latent dimensions
noise = torch.randn((n_samples, *shape), generator = generator)
z_samples = posterior.loc + posterior.scale * noise
log_ratios = (posterior.log_prob(z_samples) - prior.log_prob(z_samples)).sum(dim = -1)

# Compare the estimate with the closed form for each Gaussian
kl_mc = log_ratios.mean(dim = 0)
kl_closed = kl_to_standard_normal(mu_check, logvar_check)
absolute_error = (kl_closed - kl_mc).abs()
mc_standard_error = log_ratios.std(dim = 0) / n_samples**0.5

for i in range(n_gaussians):
    print(f'Gaussian {i + 1} | Closed form: {kl_closed[i]:.4f} | ' f'Monte Carlo: {kl_mc[i]:.4f} | Absolute error: {absolute_error[i]:.4f}')

agreement = (absolute_error <= 4 * mc_standard_error).all().item()
print(f'All estimates within four Monte Carlo standard errors: {agreement}')

Gaussian 1 | Closed form: 4.6179 | Monte Carlo: 4.6348 | Absolute error: 0.0169
Gaussian 2 | Closed form: 2.7566 | Monte Carlo: 2.7641 | Absolute error: 0.0075
Gaussian 3 | Closed form: 7.5045 | Monte Carlo: 7.4798 | Absolute error: 0.0247
Gaussian 4 | Closed form: 2.0781 | Monte Carlo: 2.0677 | Absolute error: 0.0104
Gaussian 5 | Closed form: 4.5055 | Monte Carlo: 4.4967 | Absolute error: 0.0088
All estimates within four Monte Carlo standard errors: True


### 2.5 Reconstruction Gradient Through the Latent Sample

This check measures the gradient of the reconstruction loss with respect to the weights of the encoder’s mean head. Both cases use the same batch and latent sample. Detaching the sample preserves its values while cutting the gradient path back to the encoder.

In [13]:
# Set the seed for reproducibility
torch.manual_seed(0)
# Select a batch of images
x_batch = x_train[:32]
# Encode the images
mu, logvar = vae.encode(x_batch)
z = vae.reparameterize(mu, logvar)
mean_weights = vae.mu_head.weight

# Measure the gradient through the reparameterized sample
reconstruction = reconstruction_nll(vae.decode(z), x_batch).mean()
connected_grad, = torch.autograd.grad(reconstruction, mean_weights)
connected_norm = connected_grad.norm().item()

# Repeat with the same latent values detached from the encoder
detached_reconstruction = reconstruction_nll(vae.decode(z.detach()), x_batch).mean()
detached_grad, = torch.autograd.grad(detached_reconstruction, mean_weights, allow_unused = True)
detached_norm = 0.0 if detached_grad is None else detached_grad.norm().item()

print(f'Gradient norm with reparameterization: {connected_norm:.6f}')
print(f'Gradient norm with detached sample: {detached_norm:.6f}')
print(f'Gradient check passed: {connected_norm > 0 and detached_norm == 0}')

Gradient norm with reparameterization: 0.689904
Gradient norm with detached sample: 0.000000
Gradient check passed: True


### 2.6 Negative ELBO at β = 1

With $\beta = 1$, the VAE loss should equal the negative ELBO estimated with one latent draw per image for the reconstruction term and an exact KL term. This check computes the ELBO separately and uses the same random seed for both calculations so they use the same latent draws.

In [14]:
# Select a batch of images
x_batch = x_train[:32]

# Calculate the VAE loss at beta = 1
with torch.no_grad():
    torch.manual_seed(0)
    loss_beta_1 = vae_loss(vae, x_batch, beta = 1.0)

    # Reset the seed so both forward passes use the same latent draws
    torch.manual_seed(0)
    logits, mu, logvar = vae(x_batch)

    # Compute the reconstruction log-likelihood and KL independently
    pixel_log_prob = x_batch * F.logsigmoid(logits)
    pixel_log_prob += (1 - x_batch) * F.logsigmoid(-logits)
    log_px_given_z = pixel_log_prob.sum(dim = 1)

    posterior = torch.distributions.Normal(mu, torch.exp(0.5 * logvar))
    prior = torch.distributions.Normal(torch.zeros_like(mu), torch.ones_like(mu))
    kl = torch.distributions.kl_divergence(posterior, prior).sum(dim = 1)
    negative_elbo = -(log_px_given_z - kl).mean()

# Calculate the absolute difference between the VAE loss and the negative ELBO
difference = (loss_beta_1 - negative_elbo).abs()
agreement = torch.isclose(loss_beta_1, negative_elbo, rtol = 0, atol = 1e-5)

# Print the results
print(f'VAE loss at beta = 1: {loss_beta_1.item():.6f}')
print(f'Negative ELBO: {negative_elbo.item():.6f}')
print(f'Absolute difference: {difference.item():.2e}')
print(f'ELBO check passed: {agreement.item()}')

VAE loss at beta = 1: 44.473660
Negative ELBO: 44.473656
Absolute difference: 3.81e-06
ELBO check passed: True
